In [ ]:
import numpy as np
import pandas as pd

from well.domain.identification.algorithms import ident_values, identificate, get_data_by_slices, identificate_regul, \
    get_res_dict_for_ident
from well.domain.identification.helpers import calc_rsd_by_series
from well.domain.simulation.helpers import get_dict_for_data
from well.domain.simulation.meter import Meter
from well.domain.simulation.model import construct_well, h_lim, betta_G_lim
from well.visualization.plots import build_plot, PlotData, Scale

In [ ]:
# Начальные условия
M_q = 0.5
epsilon = 0.00001
K_NUMBERS = 45000  # Количество точек моделирования
dt = 0.0001  # Суток

w_program = {2: 1.065556, 3: 0.9335, 3.7: 1}

well = construct_well(agzu_on=True, p_L_change_on=True)
well.set_w_program(w_program)


df = well.simulate(K_NUMBERS, dt, M_q, epsilon, get_dict_for_data())
df = pd.DataFrame(df)

In [ ]:
plot_data = [
             {'data': (
                PlotData(df['q'], '', 2, 'q'),
                PlotData(df['q_N'], 'r', 2, 'q_N'),
                PlotData(df['q_L'], 'k--', 2, '\\tilde{q_N}'),
              ), 'x': df['x'], 'y_scale': Scale(min=60, max=110), 'dt': dt, 'mu': 'м^3/сут'},
             {'data': (
               PlotData(df['u'], 'k', 1, '\omega'),
               PlotData(df['agzu'], 'g', 1, 'АГЗУ'),
              ), 'x': df['x'], 'y_scale': Scale(min=0, max=4), 'dt': dt, 'mu': 'ед.'},
            ]

plot = build_plot(plot_data)
plot.show()

In [ ]:
#df.to_excel("model_output.xlsx")

In [ ]:
plot_data = [
            {'data': (
              PlotData(df['p_3'], '', 2, 'p(3)'),
              PlotData(df['p_L'], 'r', 2, 'p_L'),
              PlotData(df['p_4'], 'k', 2, 'p(4)'),
              PlotData(df['p_8'], 'g', 2, 'p(8)'),
              ), 'x': df['x'], 'y_scale': Scale(min=1, max=7), 'dt': dt, 'mu': 'МПа'},

]


plot = build_plot(plot_data)
plot.show()

In [ ]:
plot_data = [
            {'data': (
              PlotData(df['p_3'], '', 2, 'p(3)'),
              PlotData(df['p_L'], 'r', 2, 'p_L'),
              PlotData(df['p_4'], 'k', 2, 'p(4)'),
              PlotData(df['p_8'], 'g', 2, 'p(8)'),
              PlotData(df['p_1'], 'g', 2, 'p(1)'),
              ), 'x': df['x'], 'y_scale': Scale(min=-1, max=6), 'dt': dt, 'mu': 'МПа'},
            {'data': (
                PlotData(df['h_4'], 'k--', 2, 'h(4)'),
                PlotData(pd.Series([h_lim]*K_NUMBERS), 'k--', 1, 'h(4)_{lim}')
              ), 'x': df['x'], 'y_scale': Scale(min=0, max=2500), 'dt': dt, 'mu': 'м.'},
            {'data': (
                PlotData(df['betta_GN'], 'g--', 2, '\\beta_{GN}'),
                PlotData(pd.Series([betta_G_lim]*K_NUMBERS), 'g--', 1, '\\beta_{GNlim}')
              ), 'x': df['x'], 'y_scale':  Scale(min=0, max=1), 'dt': dt},
]


plot = build_plot(plot_data)
plot.show()

In [ ]:
from well.visualization.plots import build_plot_interactive

plot_data = [
            {'data': (
              PlotData(df['p_3'], '', 2, 'p(3)'),
              PlotData(df['p_L'], 'r', 2, 'p_L'),
              PlotData(df['p_4'], 'k', 2, 'p(4)'),
              PlotData(df['p_8'], 'g', 2, 'p(8)'),
              PlotData(df['p_1'], 'g', 2, 'p(1)'),
              ), 'x': df['x'], 'y_scale': Scale(min=-1, max=6), 'dt': dt, 'mu': 'МПа'},

]
build_plot_interactive(plot_data)

In [ ]:
meter = Meter(dt)
meter.noise_enable = True
meter.noise_amplitude = 0.05
ident_dt, ident_k, df_ident = meter.measure(df, quant_step=None, denominator = 50)

In [ ]:
#df_ident.to_excel("model_measure.xlsx")

In [ ]:
plot_data = [
            {'data': (
              PlotData(df_ident['p_3'], '', 1, 'p(3)'),
              PlotData(df_ident['p_8'], 'g', 1, 'p(8)'),
              PlotData(df_ident['p_L'], 'r', 1, 'p_L'),
              PlotData(df_ident['p_4'], 'k--', 1, 'p(4)'),
              ), 'x': df_ident['x'], 'y_scale': Scale(min=-1, max=6), 'dt': dt, 'mu': 'МПа'},
            ]

plot = build_plot(plot_data)
plot.show()

In [ ]:
# Устойчивость идентификации

quant_step_list = [None]
denominator_list = [20]
experiment_num_max = 50

noise_list = np.arange(0, 0.5, 0.01)

res = get_res_dict_for_ident(len(quant_step_list), len(denominator_list), len(noise_list), experiment_num_max)
res_regul = get_res_dict_for_ident(len(quant_step_list), len(denominator_list), len(noise_list), experiment_num_max)

res_filterd = get_res_dict_for_ident(len(quant_step_list), len(denominator_list), len(noise_list), experiment_num_max)
res_regul_filtered = get_res_dict_for_ident(len(quant_step_list), len(denominator_list), len(noise_list), experiment_num_max)

meter = Meter(dt)
meter.noise_enable = True


def identificate_both(res, res_regul, filter_name = None):

    meter.noise_amplitude = noise
    ident_dt, ident_k, df_ident = meter.measure(df, quant_step=quant_step, denominator=denominator)

    calc_df = ident_values(ident_k, ident_dt, df_ident, well, filter_name=filter_name)
    b, _ = identificate(calc_df, ident_k)

    s_times = [(0, 1)]
    calc_df_static = get_data_by_slices(calc_df, ident_dt, s_times)

    regul_times = [(1, 4.5)]
    calc_df_regul = get_data_by_slices(calc_df, ident_dt, regul_times)
    b_regul = identificate_regul(calc_df_regul, calc_df_static, well.reservoir.v_r, m2=1, m3=3*ident_k)

    res['p_R'][i, j, k, experiment_num] = b[0]
    res['r_1'][i, j, k, experiment_num] = b[1]
    res['r_2'][i, j, k, experiment_num] = b[2]

    res_regul['p_R'][i, j, k, experiment_num] = b_regul[0]
    res_regul['r_1'][i, j, k, experiment_num] = b_regul[1]
    res_regul['r_2'][i, j, k, experiment_num] = b_regul[2]

    return calc_df, df_ident


for i, quant_step in enumerate(quant_step_list):
  for j, denominator in enumerate(denominator_list):
    for k, noise in enumerate(noise_list):
      for experiment_num in range(experiment_num_max):
        calc_df, df_ident = identificate_both(res, res_regul, filter_name=None)
        calc_df, df_ident = identificate_both(res_filterd, res_regul_filtered, filter_name='aperiodic')


In [ ]:
noise_list *= 100

In [ ]:
from matplotlib import pyplot as plt

# Визуализация
plt.figure(figsize=(16, 5))

# График давления
plt.subplot(1, 3, 1)
plt.plot(noise_list, calc_rsd_by_series(res['p_R'][0,0], well.reservoir.p_R), 'r', lw=2, label='$\sigma_{p_R}$')
plt.plot(noise_list, calc_rsd_by_series(res_regul['p_R'][0,0], well.reservoir.p_R), 'k', label='$\sigma_{p_R}$ с регуляризацией', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_filterd['p_R'][0,0], well.reservoir.p_R), 'r--', label='$\sigma_{p_R}$', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_regul_filtered['p_R'][0,0], well.reservoir.p_R), 'k--', label='$\sigma_{p_R}$ с регуляризацией', lw=2)
plt.xlabel('Амплитуда шума, %')
plt.ylabel('$\sigma$')
plt.legend()
plt.grid()


# График НЧ шума
plt.subplot(1, 3, 2)
plt.plot(noise_list, calc_rsd_by_series(res['r_1'][0,0], well.reservoir.r_1), label='$\sigma_{r(1)}$', color='r', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_regul['r_1'][0,0], well.reservoir.r_1), label='$\sigma_{r(1)}$ с регуляризацией', color='k', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_filterd['r_1'][0,0], well.reservoir.r_1), 'r--', label='$\sigma_{r(1)}$', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_regul_filtered['r_1'][0,0], well.reservoir.r_1), 'k--', label='$\sigma_{r(1)}$ с регуляризацией', lw=2)
plt.xlabel('Амплитуда шума, %')
plt.legend()
plt.grid()
#plt.ylim(0, 0.8)

# График ВЧ шума
plt.subplot(1, 3, 3)
plt.plot(noise_list, calc_rsd_by_series(res['r_2'][0,0], well.reservoir.r_2), 'r', label='$\sigma_{r(2)}$', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_regul['r_2'][0,0], well.reservoir.r_2), 'k', label='$\sigma_{r(2)}$ с регуляризацией', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_filterd['r_2'][0,0], well.reservoir.r_2), 'r--', label='$\sigma_{r(2)}$', lw=2)
plt.plot(noise_list, calc_rsd_by_series(res_regul_filtered['r_2'][0,0], well.reservoir.r_2), 'k--', label='$\sigma_{r(2)}$ с регуляризацией', lw=2)
plt.xlabel('Амплитуда шума, %')
plt.legend()
plt.grid()
#plt.ylim(0, 0.8)


plt.tight_layout()
plt.show()

In [ ]:
noise_list

In [ ]:
from well.domain.identification.helpers import error_diagrams

plt = error_diagrams(plt, res_filterd, res_regul_filtered, well)
plt.show()

In [ ]:
from matplotlib import pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))

twin1 = ax.twinx()

p2, = twin1.step(df['x'], df['p_1'], 'r', lw=2, label="$p(1)$ модель")
p3, = ax.step(df_ident['x'], calc_df['p_1_t_original'], 'g--', lw=1, label=r"$p(1)_\tau$ расчёт")
p1, = ax.step(df_ident['x'], calc_df['p_1_t'], 'b', lw=1, label=r"$p(1)_\tau$ расчёт")



ax.set(ylim=(10, 15), xlabel="Сутки", ylabel="$p(1)$")
twin1.set(ylim=(10, 15), ylabel="$p(1)$")

plt.title('Восстановление давления на забое, $МПа$')

ax.legend(handles=[p1, p2, p3])
ax.grid()
plt.show()



In [ ]:
from matplotlib import pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))

twin1 = ax.twinx()

p1, = ax.step(df_ident['x'], calc_df['q_t'], 'b', lw=1, label=r"$q_\tau$ расчёт")
p2, = ax.step(df_ident['x'], calc_df['_q_t'], 'k--', lw=2, label=r"$\^{q}_\tau$ расчёт")
p3, = twin1.plot(df['x'], df['q'], 'r--', lw=2, label="$q$ модель")


ax.set(ylim=(70, 120), xlabel="Сутки", ylabel="$q$")
twin1.set(ylim=(70, 120), ylabel=r"$q_\tau$, $\^{q}_\tau$")

plt.title('Восстановление дебитов через фильтр, $м^3/сут$')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, shadow=True, ncol=5, handles=[p1, p2, p3])
ax.grid()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))

twin1 = ax.twinx()

p1, = ax.plot(df_ident['x'], calc_df['q_L'], 'b', lw=2, label=r"$q_L$ расчёт")
p2, = ax.plot(df['x'], df['q_L'], 'r--', lw=2, label="$q_L$ модель")

ax.set(ylim=(70, 110), xlabel="Сутки", ylabel="$q_L$")
twin1.set(ylim=(70, 110), ylabel=r"$q_L$")

plt.title('Восстановление дебита насоса, $м^3/сут$')

ax.legend(handles=[p1, p2])
ax.grid()
plt.show()

In [ ]:
from matplotlib import pyplot as plt
fig, ax = plt.subplots(figsize=(10, 5))

twin1 = ax.twinx()

s_time = calc_df_static['_q_t'].index * denominator * dt

p1, = ax.step(df_ident['x'], calc_df['_q_t'], 'k--', lw=2, label=r"$\^{q}_\tau$ расчёт")
p2 = ax.scatter(s_time, calc_df_static['_q_t'], label=r"Статика")
p3, = twin1.plot(df['x'], df['q'], 'r', lw=2, label="$q$ модель")


ax.set(ylim=(60, 110), xlabel="Сутки", ylabel="$q$")
twin1.set(ylim=(60, 110), ylabel=r"$q_\tau$, $\^{q}_\tau$")

plt.title('Восстановление дебитов через фильтр, $м^3/сут$')

ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.05),
          fancybox=True, shadow=True, ncol=5, handles=[p1, p2, p3])
ax.grid()
plt.show()